In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import sys
sys.path.append('/usr0/home/naveenr/projects/spurious_concepts/ConceptBottleneck/')
sys.path.append('/usr0/home/naveenr/projects/spurious_concepts')
sys.path.append('/usr0/home/naveenr/projects/concept_decisions')

In [3]:
import torch
from sklearn.metrics import roc_auc_score
from sklearn.neural_network import MLPClassifier
import torch.nn as nn
import torch.optim as optim
import pickle
import matplotlib.pyplot as plt
import torch.nn.functional as F
from PIL import Image
from captum.attr import visualization as viz
from matplotlib.colors import LinearSegmentedColormap
import cv2
from copy import copy 
import itertools
from matplotlib.patches import Circle
import json
import argparse
import logging 
import resource
import gc 
import secrets
from torch.utils.data import Subset


In [4]:
from ConceptBottleneck.CUB.dataset import load_data

In [5]:
from src.images import *
from src.util import *
from src.models import *
from src.plot import *

## Setup Data + Model

In [6]:
torch.cuda.set_per_process_memory_fraction(0.75)
resource.setrlimit(resource.RLIMIT_AS, (30 * 1024 * 1024 * 1024, -1))
torch.set_num_threads(1)

In [7]:
train_loader, val_loader, test_loader, train_pkl, val_pkl, test_pkl = get_data(1,encoder_model="inceptionv3",dataset_name="CUB")

In [8]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'

In [9]:
joint_location = "../../models/CUB.pth"
joint_model = torch.load(joint_location,map_location='cpu')
r = joint_model.eval()

In [25]:
def run_joint_model_2(model,x,detach=True,c=None):
    """Run a joint model and get the y_pred and c_pred
    
    Arguments: 
        model: A PyTorch joint model
        x: Numpy array that we run through the model

    Returns:
        Two Torch Tensors: y_pred and c_pred
    """

    if c is None:
        output = model.first_model(x,binary=True)
    else:
        output = [i.reshape(-1,1).to(device)*5-5 for i in c] 
    output = model.forward_stage2(output)[0]
    return output 

In [26]:
run_model_function = run_joint_model_2

In [27]:
joint_model = joint_model.to(device)

## Evaluate Model

In [28]:
# test_acc =  get_accuracy(joint_model,run_model_function,test_loader)

In [29]:
total_acc = 0
n = 0

for x,y,c in test_loader:
    with torch.no_grad():
        acc = logits_to_index(run_joint_model_2(joint_model,x.to(device))).detach().cpu() == y
        acc = torch.sum(acc)
        total_acc += acc
        n += len(x)

In [30]:
total_acc/n

tensor(0.6660)

In [38]:
torch.cuda.empty_cache()

In [45]:
torch.nn.Sigmoid()(torch.Tensor([0]))

tensor([0.5000])

In [47]:
torch.logit(torch.Tensor([0.5]))

tensor([0.])

In [69]:
y

tensor([0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
        1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 2, 2, 2, 2])

tensor(0.5312)


In [20]:
test_acc 

0.6660338280980325